# 制約付きデコーディングで LLM の出力を JSON スキーマ・正規表現・選択肢に従わせる (outlines)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/13_llm/constrained_decoding.ipynb)

NLP クックブック 13-13 ／ 大規模言語モデル (LLM)

> GPU ランタイム推奨（ランタイム → ランタイムのタイプを変更 → T4 GPU）

手元の LLM に JSON や決まった形式で答えさせるとき、生成してから取り出すのではなく、生成の途中で形式に合わないトークンを禁止して、必ず形式どおりの出力を得たい。形式の違反がどれだけ減るか、内容は正しくなるのか、速度はどうかも確かめたい。

In [1]:
!pip install -q outlines==1.3.3 datasets

In [2]:
import json, re, time
from typing import Literal
import numpy as np
import pandas as pd
import outlines
import sentencepiece as spm
import torch
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from outlines.types import Regex
from pydantic import BaseModel, Field, ValidationError
from tokenizers import models
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.manual_seed(0)
name = "sbintuitions/sarashina2.2-1b-instruct-v0.1"       # SB Intuitions、MIT ライセンス
tokenizer = AutoTokenizer.from_pretrained(name)
# transformers 5 系は、このモデルの SentencePiece (Unigram) を BPE に変換してしまい、学習時と違う分割になる。
# 元の tokenizer.model から語彙とスコアを読み、分割の方式だけを Unigram に戻す（特殊トークンやチャットテンプレートはそのまま）
sp = spm.SentencePieceProcessor(model_file=hf_hub_download(name, "tokenizer.model"))
tokenizer.backend_tokenizer.model = models.Unigram(
    [(sp.id_to_piece(i), sp.get_score(i)) for i in range(sp.get_piece_size())], unk_id=sp.unk_id(), byte_fallback=True)
BF16 = torch.cuda.get_device_capability()[0] >= 8      # bfloat16 は Ampere (A100, L4) 以降。T4 では float16 を使う
model = AutoModelForCausalLM.from_pretrained(name, dtype=torch.bfloat16 if BF16 else torch.float16, device_map="auto").eval()
om = outlines.from_transformers(model, tokenizer)        # outlines (Apache 2.0) のモデル。生成は model.generate に制約の logits_processor を足して行う

def run(output_type, prompts, max_new_tokens):
    """output_type=None なら制約なし。全件を 1 回のバッチで生成し、出力と速度を返す"""
    t = time.perf_counter()
    gen = outlines.Generator(om, output_type)           # ここで形式を受理するオートマトンを作り、語彙の各トークンと対応づける
    t_build = time.perf_counter() - t
    t = time.perf_counter()
    outs = gen.batch(prompts, max_new_tokens=max_new_tokens, do_sample=False)
    elapsed = time.perf_counter() - t
    n_tok = [len(tokenizer(o, add_special_tokens=False)["input_ids"]) for o in outs]
    # バッチ生成のステップ数は、いちばん長い出力のトークン数で決まる。1 ステップあたりの時間で比べる
    return outs, {"準備の秒数": t_build, "1 件あたりの秒数": elapsed / len(prompts),
                  "出力のトークン数（平均）": np.mean(n_tok), "出力のトークン数（最大）": max(n_tok),
                  "1 ステップあたりのミリ秒": elapsed / max(n_tok) * 1000}

rng = np.random.default_rng(0)
def diff_ci(x, y, n_boot=2000):
    """同じ問題で比べる対応のあるブートストラップで、正解率の差 (x − y) と 95% 信頼区間を求める"""
    x, y = np.asarray(x, float), np.asarray(y, float)
    idx = rng.integers(0, len(x), (n_boot, len(x)))
    d = x[idx].mean(1) - y[idx].mean(1)
    return f"{(x.mean() - y.mean()) * 100:+.1f} ポイント [{np.percentile(d, 2.5) * 100:+.1f}, {np.percentile(d, 97.5) * 100:+.1f}]"

chat = lambda content: tokenizer.apply_chat_template([{"role": "user", "content": content}], add_generation_prompt=True, tokenize=False)

In [3]:
# 1. JSON スキーマ: この章の「LLM の出力から JSON を取り出して検証する」と同じ課題（レビュー → 感情・点数・観点）。
# 内容の正しさを測れるように、観点は 6 つから選ぶ形にした。レビューと正解は、このレシピのために作った 40 件
ASPECTS = ["料理", "接客", "価格", "雰囲気", "立地", "待ち時間"]
class Review(BaseModel):
    sentiment: Literal["positive", "negative", "neutral"]
    score: int = Field(ge=1, le=5)
    aspects: list[Literal["料理", "接客", "価格", "雰囲気", "立地", "待ち時間"]]

P, N, U = "positive", "negative", "neutral"
data = [
    ("料理がとても美味しくて、店員さんも親切でした。また来ます。", P, "料理 接客"), ("駅から歩いてすぐで、落ち着いた雰囲気のお店でした。", P, "立地 雰囲気"),
    ("この値段でこの味なら大満足です。", P, "価格 料理"), ("注文してから5分で料理が出てきて驚きました。味も申し分ない。", P, "待ち時間 料理"),
    ("スタッフの気配りが行き届いていて、記念日を気持ちよく過ごせました。", P, "接客"), ("窓からの夜景がきれいで、デートにぴったりの雰囲気です。", P, "雰囲気"),
    ("ランチセットが千円以下で、味も量も満足。", P, "価格 料理"), ("予約なしでもすぐに案内してもらえ、店員さんの対応も丁寧でした。", P, "待ち時間 接客"),
    ("焼きたてのパンが絶品。少し高いけれど、それだけの価値はあります。", P, "料理 価格"), ("店内は清潔で明るく、子ども連れでも入りやすかった。", P, "雰囲気"),
    ("会社の近くにあるので重宝しています。日替わり定食がおいしい。", P, "立地 料理"), ("行列ができていたが、回転が速くて10分ほどで入れた。ラーメンも最高。", P, "待ち時間 料理"),
    ("苦手な食材を伝えたら、快く対応してくれました。", P, "接客"), ("静かで落ち着けるので、一人でゆっくり食事を楽しめた。", P, "雰囲気"),
    ("料金は良心的で、店主の人柄も温かい。", P, "価格 接客"),
    ("1 時間待たされた上に、注文を間違えられた。", N, "待ち時間 接客"), ("値段のわりに量が少なくてがっかり。", N, "価格 料理"),
    ("店員の態度が悪く、二度と行きたくない。", N, "接客"), ("スープがぬるく、麺も伸びていた。", N, "料理"),
    ("駅から遠いうえに、看板がなくて迷った。", N, "立地"), ("隣の席の声がうるさく、店内も狭くて落ち着かない。", N, "雰囲気"),
    ("料理が出てくるまで40分もかかった。", N, "待ち時間"), ("この味でこの値段は高すぎる。", N, "料理 価格"),
    ("テーブルがべたべたで、店内も薄暗かった。", N, "雰囲気"), ("呼んでも店員が来ず、料理も冷めていた。", N, "接客 料理"),
    ("刺身が生臭く、食べられなかった。", N, "料理"), ("会計を間違えられたのに謝罪もなかった。", N, "接客"),
    ("駐車場がなく、最寄り駅からも徒歩20分かかる。", N, "立地"), ("予約していたのに30分も待たされ、席も狭かった。", N, "待ち時間 雰囲気"),
    ("メニューの写真と違う料理が出てきて、しかも高い。", N, "料理 価格"),
    ("可もなく不可もなく、といったところ。", U, ""), ("料理は普通。値段も相場どおり。", U, "料理 価格"),
    ("料理はおいしいが、店員の態度は残念だった。", U, "料理 接客"), ("雰囲気は良いが、料理の味は平凡だった。", U, "雰囲気 料理"),
    ("駅前で便利だが、いつも混んでいて待たされる。", U, "立地 待ち時間"), ("特に印象に残らない普通の定食屋。", U, ""),
    ("安いけれど、味はそれなり。", U, "価格 料理"), ("接客は丁寧だが、料理が出てくるのが遅い。", U, "接客 待ち時間"),
    ("量は多いが、味付けが濃すぎる。", U, "料理"), ("店内はおしゃれだが、値段が高め。", U, "雰囲気 価格"),
]
reviews, gold_sent, gold_asp = [d[0] for d in data], [d[1] for d in data], [set(d[2].split()) for d in data]

def extract_json(text):
    """「LLM の出力から JSON を取り出して検証する」と同じ: 前後の説明文やコードブロックがあっても最初の JSON オブジェクトを取り出す"""
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text.strip(), flags=re.MULTILINE)
    start = text.find("{")
    if start < 0:
        raise ValueError("JSON が見つかりません")
    return json.JSONDecoder().raw_decode(text[start:])[0]

def parse(text):
    try:
        return Review.model_validate(extract_json(text)), "OK"
    except json.JSONDecodeError:
        return None, "JSON として読めない"
    except ValueError as e:                     # ValidationError も ValueError の一種
        return None, "JSON がない" if "見つかりません" in str(e) else "スキーマに合わない"

SCHEMA = json.dumps(Review.model_json_schema(), ensure_ascii=False)
FULL = ("次の飲食店のレビューを分析し、JSON だけを出力してください。sentiment は感情（positive / negative / neutral）、score は 1〜5 の評価、"
        f"aspects はレビューが触れている観点を {'、'.join(ASPECTS)} から選んだリストです。\nスキーマ: {SCHEMA}\n\nレビュー: ")
SHORT = "次の飲食店のレビューを分析して、JSON で答えてください。\n\nレビュー: "    # スキーマも観点の候補も書かない短い指示

def evaluate(outs):
    parsed = [parse(o) for o in outs]
    ok = [r for r, s in parsed if r]
    tp = sum(len(set(r.aspects) & g) for (r, _), g in zip(parsed, gold_asp) if r)
    n_pred, n_gold = sum(len(set(r.aspects)) for r in ok), sum(len(g) for g in gold_asp)
    return parsed, {"形式の違反": 1 - len(ok) / len(outs),
                    "感情の正解率": np.mean([bool(r) and r.sentiment == g for (r, _), g in zip(parsed, gold_sent)]),   # 違反は不正解として数える
                    "観点の F1": 2 * tp / (n_pred + n_gold),
                    "感情と点数の矛盾": np.mean([r.sentiment == P and r.score <= 2 or r.sentiment == N and r.score >= 4 for r in ok])}

# 各条件を 40 件まとめて生成する（Colab の T4 で 1〜2 分程度）
results, parsed_all = {}, {}     # parsed_all[条件] は（出力, Review か None, 状態）のリスト
for label, prefix, output_type in [("生成して取り出す", FULL, None), ("制約あり", FULL, Review), ("制約あり（短い指示）", SHORT, Review)]:
    outs, speed = run(output_type, [chat(prefix + r) for r in reviews], max_new_tokens=150)
    parsed, metrics = evaluate(outs)
    results[label] = {**metrics, **speed}
    parsed_all[label] = [(o, r, s) for o, (r, s) in zip(outs, parsed)]
pd.set_option("display.width", 200)
print(pd.DataFrame(results).round(3).to_string())
correct = {l: [bool(r) and r.sentiment == g for (_, r, _), g in zip(parsed_all[l], gold_sent)] for l in results}
print("感情の正解率の差（制約あり − 生成して取り出す）:", diff_ci(correct["制約あり"], correct["生成して取り出す"]))
for label in results:
    print(f"--- {label}: 違反の内訳", pd.Series([s for _, _, s in parsed_all[label]]).value_counts().to_dict())
    print("    例:", [o for o, _, s in parsed_all[label] if s != "OK"][:2] or parsed_all[label][0][0])

               生成して取り出す     制約あり  制約あり（短い指示）
形式の違反             0.100    0.000       0.400
感情の正解率            0.725    0.775       0.400
観点の F1            0.656    0.759       0.550
感情と点数の矛盾          0.000    0.000       0.000
準備の秒数             0.000    0.629       0.725
1 件あたりの秒数         0.104    0.189       0.047
出力のトークン数（平均）     34.850   16.350      19.250
出力のトークン数（最大）    109.000   18.000      26.000
1 ステップあたりのミリ秒    38.208  420.511      72.951
感情の正解率の差（制約あり − 生成して取り出す）: +5.0 ポイント [+0.0, +12.5]
--- 生成して取り出す: 違反の内訳 {'OK': 36, 'スキーマに合わない': 4}
    例: ['{\n  "sentiment": "positive",\n  "score": 5,\n  "aspects": ["料理", "味"]\n}', '{\n  "sentiment": "positive",\n  "score": 4,\n  "aspects": ["雰囲気", "子ども連れfriendly"]\n}\n\nこのレビューは、全体的な印象が良く、特に「清潔で明るい店内」と「子ども連れでも入りやすかった」という点が高く評価されています。これにより、全体的にポジティブな感情（positive）と中程度の評価（score=4）を示しています。 aspects セクションでは、「雰囲気」と「子ども連れfriendly」という観点が特に言及されており、これが全体の評価を高めています。']
--- 制約あり: 違反の内訳 {'OK': 40}
    例: {"sentiment":"positive","score":5,"aspects":["料理","接客"

In [4]:
# 感情の誤りの中身: 形式は守れていても、内容が正しいとは限らない
rows = []
for i, (rv, g) in enumerate(zip(reviews, gold_sent)):
    answers = []
    for label in results:
        _, r, status = parsed_all[label][i]
        answers.append(r.sentiment if r else status)    # 違反なら違反の種類を表示する
    rows.append((rv, g, *answers))
table = pd.DataFrame(rows, columns=["レビュー", "正解", *results])
print(table[table["制約あり"] != table["正解"]].to_string(max_colwidth=30))
print(pd.crosstab(table["正解"], table["制約あり"]).to_string())

                      レビュー       正解   生成して取り出す      制約あり 制約あり（短い指示）
31         料理は普通。値段も相場どおり。  neutral   negative  negative    neutral
32   料理はおいしいが、店員の態度は残念だった。  neutral   negative  negative   negative
33     雰囲気は良いが、料理の味は平凡だった。  neutral  スキーマに合わない  negative   negative
34  駅前で便利だが、いつも混んでいて待たされる。  neutral   negative  negative   negative
35        特に印象に残らない普通の定食屋。  neutral   negative  negative   negative
36           安いけれど、味はそれなり。  neutral   negative  negative   negative
37    接客は丁寧だが、料理が出てくるのが遅い。  neutral   negative  negative   negative
38         量は多いが、味付けが濃すぎる。  neutral  スキーマに合わない  negative   negative
39        店内はおしゃれだが、値段が高め。  neutral   negative  negative   negative
制約あり      negative  neutral  positive
正解                                   
negative        15        0         0
neutral          9        1         0
positive         0        0        15


In [5]:
# 2. 選択肢と正規表現
# 選択肢: Literal を渡すと、出力はそのどれか 1 つになる（同じ 40 件の感情だけを答えさせる）
ask = "次の飲食店のレビューの感情を positive、negative、neutral のどれか 1 語で答えてください。\n\nレビュー: "
for label, output_type in [("制約なし", None), ("選択肢", Literal["positive", "negative", "neutral"])]:
    outs, speed = run(output_type, [chat(ask + r) for r in reviews], max_new_tokens=20)
    print(f"{label}: 選択肢どおりの出力 {np.mean([o.strip() in (P, N, U) for o in outs]):.2f} / "
          f"正解率 {np.mean([o.strip() == g for o, g in zip(outs, gold_sent)]):.3f} / 例 {outs[:3]}")

# 正規表現: 文中の日付を YYYY-MM-DD に直させる。日付のない文（後ろの 4 件）を混ぜておく
dates = [("2024年3月5日に予約しました。", "2024-03-05"), ("会議は2023年11月20日の午後に開かれる。", "2023-11-20"),
         ("令和6年4月1日から新しい制度が始まる。", "2024-04-01"), ("平成31年4月30日に退位の儀式が行われた。", "2019-04-30"),
         ("2022年の大晦日に家族で集まった。", "2022-12-31"), ("締め切りは2025年1月10日です。", "2025-01-10"),
         ("昭和64年1月7日は昭和最後の日である。", "1989-01-07"), ("2020年2月29日はうるう日だった。", "2020-02-29"),
         ("明日は雨が降るらしい。", "不明"), ("来週の会議の場所が変わった。", "不明"),
         ("この店の料理は本当においしい。", "不明"), ("毎年春になると桜を見に行く。", "不明")]
ask = "次の文に書かれている日付を YYYY-MM-DD の形式で答えてください。和暦は西暦に直してください。日付が書かれていなければ「不明」と答えてください。\n\n文: "
DATE = r"\d{4}-(0[1-9]|1[0-2])-(0[1-9]|[12]\d|3[01])"
for label, output_type in [("制約なし", None), ("日付だけ", Regex(DATE)), ("日付か「不明」", Regex(f"{DATE}|不明"))]:
    outs, speed = run(output_type, [chat(ask + s) for s, _ in dates], max_new_tokens=60)
    found = [m.group(0) if (m := re.search(f"{DATE}|不明", o)) else None for o in outs]   # 制約なしは出力から取り出す
    print(f"{label}: 形式どおり {np.mean([re.fullmatch(f'{DATE}|不明', o.strip()) is not None for o in outs]):.2f} / "
          f"正解（日付あり 8 件）{sum(f == g for f, (_, g) in zip(found[:8], dates))} / 正解（日付なし 4 件）{sum(f == '不明' for f in found[8:])}")
    print("   ", [f"{s[:8]}→{o.strip()[:20]}" for (s, _), o in zip(dates, outs)])

制約なし: 選択肢どおりの出力 0.97 / 正解率 0.800 / 例 ['positive', 'positive', 'positive']
選択肢: 選択肢どおりの出力 1.00 / 正解率 0.800 / 例 ['positive', 'positive', 'positive']
制約なし: 形式どおり 0.00 / 正解（日付あり 8 件）3 / 正解（日付なし 4 件）2
    ['2024年3月5→与えられた文「2024年3月5日に予約し', '会議は2023年→与えられた文に書かれている日付は「202', '令和6年4月1日→与えられた文から、令和6年4月1日から新', '平成31年4月3→与えられた文に書かれている日付は「平成3', '2022年の大晦→与えられた文から、2022年の大晦日に家', '締め切りは202→与えられた文に書かれている日付は「202', '昭和64年1月7→昭和64年1月7日は、昭和天皇の崩御によ', '2020年2月2→与えられた文から、2020年2月29日は', '明日は雨が降るら→与えられた文「明日は雨が降るらしい」から', '来週の会議の場所→与えられた文から、来週の会議の場所は「 ', 'この店の料理は本→与えられた文から読み取れる情報をもとに、', '毎年春になると桜→毎年春になると桜を見に行く。\n\n(和暦を']
日付だけ: 形式どおり 1.00 / 正解（日付あり 8 件）7 / 正解（日付なし 4 件）0
    ['2024年3月5→2024-03-05', '会議は2023年→2023-11-20', '令和6年4月1日→2024-04-01', '平成31年4月3→2019-04-30', '2022年の大晦→2022-12-31', '締め切りは202→2025-01-10', '昭和64年1月7→1989-01-08', '2020年2月2→2020-02-29', '明日は雨が降るら→2023-05-14', '来週の会議の場所→2023-09-15', 'この店の料理は本→2023-05-15', '毎年春になると桜→2023-04-01']
日付か「不明」: 形式どおり 1.00 / 正解（日付あり 8 件）7 / 正解（日付なし 4 件）3
    ['2024年3月5→2024

In [6]:
# 3. 強すぎる制約: 算数の文章題で、答えだけの JSON に縛ると考える過程を書けなくなる
# MGSM (Shi et al., 2022) の日本語の先頭 50 問。CC BY 4.0（Hugging Face のメタデータは CC BY-SA 4.0）。
# この章の「考える過程を書かせて LLM に算数の文章題を解かせる（Chain-of-Thought と self-consistency）」と同じデータ
mgsm = load_dataset("juletxara/mgsm", "ja", split="test").select(range(50))
gold = np.array(mgsm["answer_number"], dtype=float)

class Answer(BaseModel):
    answer: int
class ReasonThenAnswer(BaseModel):
    reasoning: str
    answer: int
class AnswerThenReason(BaseModel):
    answer: int
    reasoning: str

THINK = "次の問題に答えてください。ステップごとに考えて途中の計算を書き、最後に答えを書いてください。"
def last_number(text):
    found = re.findall(r"-?\d+(?:\.\d+)?", text.replace(",", ""))
    return float(found[-1]) if found else np.nan

def answer_of(text, schema):
    if schema is None:                        # 制約なしは、出力の最後に出てきた数値を答えとする
        return last_number(text)
    try:
        return float(schema.model_validate_json(text).answer)
    except ValidationError:                   # 上限のトークン数で打ち切られて JSON が閉じていない
        return np.nan

# 4 条件 × 50 問（Colab の T4 で 4〜6 分程度）
math_res, hits = {}, {}
for label, schema in [("制約なし (CoT)", None), ("answer だけ", Answer), ("reasoning → answer", ReasonThenAnswer), ("answer → reasoning", AnswerThenReason)]:
    hint = "" if schema is None else f"\n出力は次の JSON スキーマに従ってください: {json.dumps(schema.model_json_schema(), ensure_ascii=False)}"
    outs, speed = run(schema, [chat(f"{THINK}{hint}\n\n問題: {q}") for q in mgsm["question"]], max_new_tokens=400)
    pred = np.array([answer_of(o, schema) for o in outs])
    hits[label] = pred == gold
    math_res[label] = {"正解率": (pred == gold).mean(), "答えが取れない": np.isnan(pred).mean(), **speed}
    print(f"--- {label}（正解 {gold[1]:g}）:", outs[1][:200].replace("\n", " "))
print(pd.DataFrame(math_res).round(3).to_string())
for label in list(math_res)[1:]:
    print(f"正解率の差（{label} − 制約なし (CoT)）:", diff_ci(hits[label], hits["制約なし (CoT)"]))
print(f"GPU メモリ最大 {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")

--- 制約なし (CoT)（正解 3）: まず、青色の繊維が2巻必要であることが与えられています。次に、白色の繊維は青色の繊維の半分を用いるので、青色の繊維の半分を計算する必要があります。青色の繊維が2巻なので、白色の繊維は2巻の半分、つまり1巻が必要となります。  したがって、ローブを作成するために必要な繊維の総巻数は、青色の繊維と白色の繊維の合計です。つまり、2巻（青色） + 1巻（白色） = 3巻となります。  よって、全体で必要な
--- answer だけ（正解 3）: {"answer":3}
--- reasoning → answer（正解 3）: {"reasoning":": まず、青色の繊維が2巻必要であることがわかりました。次に、白色の繊維は青色の繊維の半分を使うので、2巻の半分、つまり1巻が必要です。したがって、全体では青色の繊維2巻と白色の繊維1巻、合計3巻の繊維が必要となります。","answer":3}
--- answer → reasoning（正解 3）: {"answer":2,"reasoning":"必要な青色の繊維の量を計算します。問題文では、青色の繊維を2巻分用いると指定されています。そして、白色の繊維はその半分が必要であるとされています。したがって、青色の繊維の半分は1巻分です。よって、全体で2巻（青色）+ 1巻（白色）= 3巻の繊維が必要となります。"}
               制約なし (CoT)  answer だけ  reasoning → answer  answer → reasoning
正解率                 0.360      0.060               0.140               0.040
答えが取れない             0.000      0.000               0.020               0.000
準備の秒数               0.000      0.308               0.594               0.311
1 件あたりの秒数           0.489      0.092               2.357               0.852
出

- 制約付きデコーディングは、生成の各ステップで、形式（JSON スキーマ、正規表現、選択肢）に合わない続きになるトークンの確率を 0 にしてから次のトークンを選ぶ方法である。outlines（dottxt、Apache 2.0）は、形式を正規表現（有限オートマトン）に直し、語彙の各トークンがどの状態から受理されるかを前もって索引にしておく (Willard and Louf, 2023)。transformers のモデルでは `model.generate` に logits_processor を足すだけなので、生成の設定（貪欲法、`max_new_tokens` など）はそのまま使える。ここでは版を 1.3.3 に固定した。裏側の実装は xgrammar (Apache 2.0) や llguidance (MIT) にも切り替えられる。lm-format-enforcer (MIT) も同じ目的のライブラリである。
- JSON（40 件）: 生成してから取り出す方式では 10%（4 件）がスキーマに合わなかった（表示した例は、観点に候補にない「味」「子ども連れfriendly」を書いたもの）。制約ありでは違反は 0 件で、観点の F1 も 0.656 から 0.759 に上がった。感情の正解率は 72.5% と 77.5% で、差の +5.0 ポイント [+0.0, +12.5] は違反を不正解に数えた分にあたり、この件数では偶然と区別しにくい。制約ありの出力は改行のない JSON だけ（平均 16.4 トークン）で、取り出す方式は改行や説明が付いて平均 34.9 トークンだった。
- 制約で守られるのは形式だけである。良い点と悪い点が混じった neutral の 10 件のうち、制約ありは 9 件を negative と答えた（取り出す方式も 7 件が negative、2 件が違反）。指示を短くしてスキーマを書かないと、制約ありでも 40%（16 件）の出力が `score` に 8 や 10 を入れてスキーマに合わず、違反を不正解に数えた感情の正解率は 40% に下がった（形式どおりの 24 件では 16 件が正解）。制約があるので、この 16 件はどれも `score` の範囲だけが違反である。outlines_core 0.2.14 は、JSON スキーマの `minimum` / `maximum`（pydantic の `ge` / `le`）を正規表現に反映しない（`build_regex_from_schema` で確かめた）。制約を掛けても pydantic での検証は残し、範囲を必ず守らせたいなら `Literal[1, 2, 3, 4, 5]` のような列挙にする。指示とスキーマの説明はプロンプトにも書く。
- 選択肢: 感情だけを答えさせると、制約なしでも 97% が選択肢どおりの 1 語で、正解率は制約の有無によらず 80.0% だった。選択肢の制約は、各ステップで許されたトークンのうち最も確率の高いものを選ぶだけで、選択肢の全体の尤度を比べるわけではない（この章の「選択肢の尤度を比べて LLM に分類させる」とは結果が違いうる）。
- 正規表現（12 文）: 制約なしでは「与えられた文に書かれている日付は…」のように説明から始まり、YYYY-MM-DD だけの出力は 0 件、日付のある 8 文から取り出せた正解は 3 件だった。日付の形式だけを許すと 8 文中 7 文が正解になったが、日付のない 4 文すべてに、文に書かれていない日付（2023-05-14 など）を作った。「不明」も許すと 4 文中 3 文で「不明」と答えた（「毎年春になると…」は 2023-04-01）。昭和64年1月7日は、どちらでも形式は正しいが 1 日ずれた 1989-01-08 になった。答えがない場合の逃げ道を形式に入れておかないと、制約がもっともらしい誤りを作らせる。
- 強すぎる制約（MGSM の 50 問）: 自由に考えさせて最後の数値を取ると正解率 36%、答えだけの JSON に縛ると 6%（差 −30.0 ポイント [−44.0, −16.0]）だった。同じく `answer` を先に、`reasoning` を後に置くと 4% で、例の問題のように、先に書いた答え (2) と後の推論 (3) が食い違う。`reasoning` を先に置くと 14% まで戻るが、自由な CoT よりはまだ低かった（−22.0 ポイント [−36.0, −6.0]。原因は調べていない）。形式の制約で推論の課題の性能が下がることは Tam et al. (2024) も報告している。推論が要る課題では、自由に考えさせたあとで別の呼び出しで構造化するか、推論のフィールドを先に置く。
- 速度: バッチ生成では、ステップ数がいちばん長い出力のトークン数で決まるので、表の「1 ステップあたりのミリ秒」で比べる。出力の最大の長さがほぼ同じ MGSM の 2 条件では、`reasoning` → `answer` が 295 ミリ秒、自由な CoT が 61 ミリ秒で、制約ありは 1 ステップが約 5 倍遅かった。各ステップで、バッチの 1 件ずつ索引から許されるトークンのマスクを作り、GPU に送って logits に掛ける処理が加わるためと考えられる（内訳は測っていない）。出力が短い条件はプロンプトの処理などの固定の時間が大きく、1 ステップあたりでは比べにくいが、出力が短くなる分、1 件あたりでは速くなりうる（短い指示の JSON は 0.047 秒、取り出す方式は 0.104 秒）。形式ごとの準備（オートマトンの構築）は 0.3〜0.7 秒だった。時間は共有の GPU で測った目安である。GPU メモリは最大 5.7 GB（T4 と同じ float16 で実行した値）。
- データは、レビュー 40 件と日付の 12 文をこのレシピのために作り、正解も付けた（neutral の判断などには異論がありうる）。MGSM は先頭の 50 問だけなので、信頼区間の幅は 30 ポイント近くある。モデルは sarashina2.2-1b-instruct（SB Intuitions、MIT ライセンス）。API のモデルでは、LLM の API の章の「Gemini に決まった形式（JSON スキーマ）で出力させる」「Claude に決まった形式（JSON スキーマ）で出力させる」の機能を使う。トークナイザーの修正については、この章の「オープンな LLM とチャットする（チャットテンプレート）」の注意書きを参照。